# 4.2 — Predict Drilling Mode

## Objective

This notebook develops a machine learning regression model to predict the
drilling operating mode from drilling sensor and operational parameters.

The target variable is:

`drill_mode`

The model is evaluated using:

- Mean Absolute Error (MAE)
- Root Mean Squared Error (RMSE)
- R² Score

A chronological train/test split is used to preserve the temporal nature of
the drilling data and prevent future observations from being used to predict
earlier observations.

Multiple machine learning models are compared, followed by automatic
selection of the best-performing model.

In [49]:
# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import os
import time
import json
import warnings
import joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from sklearn.ensemble import RandomForestRegressor

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

warnings.filterwarnings("ignore")

print("Libraries imported successfully.")

Libraries imported successfully.


## 2. Load Dataset

The cleaned drilling dataset is loaded for modeling.

The dataset must contain the `timestamp` column, the drilling features, and
the `drill_mode` target.

In [50]:
# ============================================================
# 2. LOAD DATASET
# ============================================================

csv_path = "drilling_data_cleaned.csv"

df = pd.read_csv(
    csv_path
)

print(
    "Dataset shape:",
    df.shape
)

display(
    df.head()
)

Dataset shape: (598709, 37)


,block_position,weight_on_bit,hookload,slips_set,rop_depth_hour,on_bottom,top_drive_rpm,top_drive_torque_ft_lbs,flow_in,pump_pressure,...,depth_hole_tvd,differential_pressure,downhole_torque,drill_mode,year,month,day,hour,day_of_week,timestamp
0,43.55661,46.948985,123.68008,0.0,0.0,0.0,0.31182,0.0,57.19964,40.23666,...,7389.89,-531.16685,1894.27245,0.0,2020,10,25,12,6,2020-10-25 12:00:00
1,43.55661,46.948985,123.68008,0.0,0.0,0.0,0.31182,0.0,57.19964,40.23666,...,7389.89,-531.16685,1894.27245,0.0,2020,10,25,12,6,2020-10-25 12:00:00
2,43.55661,46.948985,123.68008,0.0,0.0,0.0,0.31182,0.0,57.19964,40.23666,...,7389.89,-531.16685,1894.27245,0.0,2020,10,25,12,6,2020-10-25 12:00:00
3,43.55661,46.948985,123.68008,0.0,0.0,0.0,0.31182,0.0,0.00000,40.23666,...,7389.89,-531.16685,1894.27245,0.0,2020,10,25,12,6,2020-10-25 12:00:00
4,43.55661,46.948985,123.68008,0.0,0.0,0.0,0.31182,0.0,0.00000,40.23666,...,7389.89,-531.16685,1894.27245,0.0,2020,10,25,12,6,2020-10-25 12:00:00


## 3. Timestamp Preparation

The timestamp is converted to datetime format and the observations are
sorted chronologically.

A chronological split will later be used so that future observations are
never used to train the model.

In [51]:
# ============================================================
# 3. TIMESTAMP PREPARATION
# ============================================================

if "timestamp" not in df.columns:

    raise ValueError(
        "timestamp column not found."
    )

df["timestamp"] = pd.to_datetime(
    df["timestamp"],
    errors="coerce"
)

df = (
    df
    .dropna(
        subset=["timestamp"]
    )
    .sort_values("timestamp")
    .reset_index(drop=True)
)

print(
    "Time range:",
    df["timestamp"].min(),
    "→",
    df["timestamp"].max()
)

print(
    "Rows:",
    len(df)
)

Time range: 2020-10-25 12:00:00 → 2021-01-05 11:00:00
Rows: 598709


## 4. Target Verification

`drill_mode` is used as the numerical regression target.

The target distribution is inspected before modeling to identify missing
values and the available mode values.

In [52]:
# ============================================================
# 4. TARGET VERIFICATION
# ============================================================

target = "drill_mode"

if target not in df.columns:

    raise ValueError(
        f"Target '{target}' not found."
    )

print("=" * 70)
print("DRILL MODE TARGET")
print("=" * 70)

print(
    "\nMissing values:",
    df[target].isna().sum()
)

print(
    "\nUnique values:"
)

print(
    df[target]
    .value_counts(
        dropna=False
    )
    .sort_index()
)

print(
    "\nStatistics:"
)

display(
    df[target].describe()
)

DRILL MODE TARGET

Missing values: 45

Unique values:
drill_mode
0.0    568024
1.0         5
2.0     30635
NaN        45
Name: count, dtype: int64

Statistics:


count    598664.000000
mean          0.102353
std           0.440706
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max           2.000000
Name: drill_mode, dtype: float64

## 5. Candidate Features

The candidate features consist of drilling sensor measurements and
operational parameters.

The target variable is explicitly excluded from the model feature set.

In [53]:
# ============================================================
# 5. CANDIDATE FEATURES
# ============================================================

candidate_features = [
    "block_position",
    "weight_on_bit",
    "hookload",
    "slips_set",
    "rop_depth_hour",
    "on_bottom",
    "top_drive_rpm",
    "top_drive_torque_ft_lbs",
    "flow_in",
    "pump_pressure",
    "spm_total",
    "pit_volume_active",
    "pit_gl_active",
    "gas_total_units",
    "trip_volume_active",
    "trip_gl",
    "return_flow",
    "rig_mode",
    "rockit_on_off",
    "mwd_inclination",
    "mwd_azimuth",
    "mud_temp",
    "h2s_01",
    "rig_event_code",
    "total_depth",
    "bit_diameter",
    "bit_rpm",
    "depth_hole_tvd",
    "differential_pressure",
    "downhole_torque"
]

available_features = [
    feature
    for feature in candidate_features
    if feature in df.columns
]

missing_features = [
    feature
    for feature in candidate_features
    if feature not in df.columns
]

print("Available features:")

for feature in available_features:
    print(
        f"  - {feature}"
    )

print("\nMissing features:")

for feature in missing_features:
    print(
        f"  - {feature}"
    )

Available features:
  - block_position
  - weight_on_bit
  - hookload
  - slips_set
  - rop_depth_hour
  - on_bottom
  - top_drive_rpm
  - top_drive_torque_ft_lbs
  - flow_in
  - pump_pressure
  - spm_total
  - pit_volume_active
  - pit_gl_active
  - gas_total_units
  - trip_volume_active
  - trip_gl
  - return_flow
  - rig_mode
  - rockit_on_off
  - mwd_inclination
  - mwd_azimuth
  - mud_temp
  - h2s_01
  - rig_event_code
  - total_depth
  - bit_diameter
  - bit_rpm
  - depth_hole_tvd
  - differential_pressure
  - downhole_torque

Missing features:


## 6. Feature Leakage Check

The target `drill_mode` must not appear in the predictor variables.

Timestamp and calendar fields are also excluded from direct modeling.

In [54]:
# ============================================================
# 6. FEATURE FILTERING
# ============================================================

excluded_features = [
    "timestamp",
    "year",
    "month",
    "day",
    "hour",
    "day_of_week",
    "drill_mode"
]

model_features = [
    feature
    for feature in available_features
    if feature not in excluded_features
]

if target in model_features:

    raise ValueError(
        "DATA LEAKAGE: drill_mode is present "
        "in the feature list."
    )

print(
    "Target in feature list?",
    target in model_features
)

print(
    "\nNumber of model features:",
    len(model_features)
)

Target in feature list? False

Number of model features: 30


## 7. Prepare Modeling Data

The modeling dataset contains the timestamp, target, and selected predictor
variables.

Rows with missing target values are removed because a supervised model
cannot be trained without a target.

In [55]:
# ============================================================
# 7. PREPARE MODELING DATA
# ============================================================

model_df = df[
    ["timestamp", target]
    +
    model_features
].copy()

model_df = model_df.dropna(
    subset=[target]
)

X = model_df[
    model_features
].copy()

y = model_df[
    target
].copy()

timestamps = model_df[
    "timestamp"
].copy()

print(
    "X shape:",
    X.shape
)

print(
    "y shape:",
    y.shape
)

X shape: (598664, 30)
y shape: (598664,)


## 8. Temporal Train/Test Split

The earliest 70% of observations are used for model development and the
latest 30% are reserved as an untouched chronological test set.

The test period is not used for hyperparameter selection.

In [56]:
# ============================================================
# 8A. DRILLING MODE DISTRIBUTION OVER TIME
# ============================================================

mode_distribution = (
    model_df
    .groupby(
        model_df["timestamp"].dt.date
    )[target]
    .agg(
        ["count", "nunique"]
    )
)

print("=" * 70)
print("DRILLING MODE DISTRIBUTION BY DATE")
print("=" * 70)

display(
    mode_distribution[
        mode_distribution["nunique"] > 1
    ].tail(20)
)

print(
    "\nDates containing both drilling modes:",
    (
        mode_distribution["nunique"] > 1
    ).sum()
)

DRILLING MODE DISTRIBUTION BY DATE


,count,nunique
timestamp,,
2020-10-31,7721,3
2020-11-01,8892,2
2020-11-02,8546,2
2020-11-03,7718,2
2020-11-04,8032,2



Dates containing both drilling modes: 5


In [57]:
# ============================================================
# 8. EVENT-BASED CHRONOLOGICAL TRAIN / TEST SPLIT
# ============================================================

print("=" * 70)
print("EVENT-BASED CHRONOLOGICAL SPLIT")
print("=" * 70)

train_end = pd.Timestamp(
    "2020-11-02 23:59:59"
)

test_start = pd.Timestamp(
    "2020-11-03 00:00:00"
)

test_end = pd.Timestamp(
    "2020-11-04 23:59:59"
)

train_mask = (
    model_df["timestamp"]
    <= train_end
)

test_mask = (
    (model_df["timestamp"] >= test_start)
    &
    (model_df["timestamp"] <= test_end)
)

X_train = X.loc[
    train_mask
].copy()

X_test = X.loc[
    test_mask
].copy()

y_train = y.loc[
    train_mask
].copy()

y_test = y.loc[
    test_mask
].copy()

time_train = timestamps.loc[
    train_mask
].copy()

time_test = timestamps.loc[
    test_mask
].copy()


print(
    "\nTraining rows:",
    len(X_train)
)

print(
    "Testing rows:",
    len(X_test)
)

print(
    "\nTraining period:"
)

print(
    time_train.min(),
    "→",
    time_train.max()
)

print(
    "\nTesting period:"
)

print(
    time_test.min(),
    "→",
    time_test.max()
)

print(
    "\nTraining target distribution:"
)

display(
    y_train.value_counts(
        normalize=True
    ).sort_index()
)

print(
    "\nTesting target distribution:"
)

display(
    y_test.value_counts(
        normalize=True
    ).sort_index()
)

EVENT-BASED CHRONOLOGICAL SPLIT

Training rows: 62698
Testing rows: 15750

Training period:
2020-10-25 12:00:00 → 2020-11-02 23:00:00

Testing period:
2020-11-03 00:00:00 → 2020-11-04 23:00:00

Training target distribution:


drill_mode
0.0    0.680915
1.0    0.000080
2.0    0.319005
Name: proportion, dtype: float64


Testing target distribution:


drill_mode
0.0    0.324825
2.0    0.675175
Name: proportion, dtype: float64

In [58]:
# ============================================================
# 8A. TARGET VARIATION VALIDATION
# ============================================================

print("=" * 70)
print("TARGET VARIATION CHECK")
print("=" * 70)

print(
    "Training unique modes:",
    sorted(y_train.unique())
)

print(
    "Testing unique modes:",
    sorted(y_test.unique())
)

if y_train.nunique() < 2:

    raise ValueError(
        "Training set contains only one "
        "drill_mode value. "
        "The split must be adjusted."
    )

if y_test.nunique() < 2:

    print(
        "WARNING: Test set contains only one "
        "drill_mode value. R² may not be meaningful."
    )

else:

    print(
        "\n✓ Both drilling modes are present "
        "in the test period."
    )

print(
    "\n✓ Training target variation:",
    y_train.nunique(),
    "unique values"
)

print(
    "✓ Test target variation:",
    y_test.nunique(),
    "unique values"
)

TARGET VARIATION CHECK
Training unique modes: [np.float64(0.0), np.float64(1.0), np.float64(2.0)]
Testing unique modes: [np.float64(0.0), np.float64(2.0)]

✓ Both drilling modes are present in the test period.

✓ Training target variation: 3 unique values
✓ Test target variation: 2 unique values


In [59]:
# ============================================================
# 8B. CHECK DRILLING-MODE DATA BY DATE
# ============================================================

daily_distribution = (
    model_df
    .assign(
        date=model_df["timestamp"].dt.date
    )
    .groupby(
        "date"
    )[target]
    .value_counts()
    .unstack(
        fill_value=0
    )
)

print("=" * 70)
print("DAILY DRILLING MODE COUNTS")
print("=" * 70)

display(
    daily_distribution
)

DAILY DRILLING MODE COUNTS


drill_mode,0.0,1.0,2.0
date,,,
2020-10-25,2290,0,0
2020-10-26,4204,0,0
2020-10-27,5818,0,0
2020-10-28,8350,0,0
2020-10-29,8630,0,0
...,...,...,...
2021-01-01,8293,0,0
2021-01-02,8229,0,0
2021-01-03,8192,0,0


In [60]:
# ============================================================
# 8C. CHECK AVAILABLE TARGET VALUES
# ============================================================

print("=" * 70)
print("TARGET VALUES BY DATE")
print("=" * 70)

for date, group in (
    model_df
    .groupby(
        model_df["timestamp"].dt.date
    )
):

    print(
        f"{date}: "
        f"modes={sorted(group[target].unique())}, "
        f"rows={len(group)}"
    )

TARGET VALUES BY DATE
2020-10-25: modes=[np.float64(0.0)], rows=2290
2020-10-26: modes=[np.float64(0.0)], rows=4204
2020-10-27: modes=[np.float64(0.0)], rows=5818
2020-10-28: modes=[np.float64(0.0)], rows=8350
2020-10-29: modes=[np.float64(0.0)], rows=8630
2020-10-30: modes=[np.float64(0.0)], rows=8247
2020-10-31: modes=[np.float64(0.0), np.float64(1.0), np.float64(2.0)], rows=7721
2020-11-01: modes=[np.float64(0.0), np.float64(2.0)], rows=8892
2020-11-02: modes=[np.float64(0.0), np.float64(2.0)], rows=8546
2020-11-03: modes=[np.float64(0.0), np.float64(2.0)], rows=7718
2020-11-04: modes=[np.float64(0.0), np.float64(2.0)], rows=8032
2020-11-05: modes=[np.float64(0.0)], rows=8425
2020-11-06: modes=[np.float64(0.0)], rows=8279
2020-11-07: modes=[np.float64(0.0)], rows=8387
2020-11-08: modes=[np.float64(0.0)], rows=8260
2020-11-09: modes=[np.float64(0.0)], rows=8405
2020-11-10: modes=[np.float64(0.0)], rows=7981
2020-11-11: modes=[np.float64(0.0)], rows=8498
2020-11-12: modes=[np.float64(

In [61]:
mode_timeline = (
    model_df
    .groupby(
        model_df["timestamp"].dt.date
    )[target]
    .agg(
        total_rows="count",
        unique_modes="nunique",
        mean_mode="mean",
        min_mode="min",
        max_mode="max"
    )
)

display(
    mode_timeline[
        mode_timeline["max_mode"] == 1
    ]
)

,total_rows,unique_modes,mean_mode,min_mode,max_mode
timestamp,,,,,


In [62]:
daily_mode = (
    model_df
    .assign(
        date=model_df["timestamp"].dt.date
    )
    .groupby("date")[target]
    .mean()
)

display(
    daily_mode[
        daily_mode.ne(
            daily_mode.shift()
        )
    ].to_frame("mean_drill_mode")
)

,mean_drill_mode
date,
2020-10-25,0.000000
2020-10-31,0.703147
2020-11-01,1.983581
2020-11-02,1.982214
2020-11-03,1.984193
2020-11-04,0.741285
2020-11-05,0.000000


## 9. Missing Value Handling

Median imputation is fitted exclusively on the training data.

The fitted imputer is then applied to the chronological test data.

In [63]:
# ============================================================
# 9. IMPUTATION
# ============================================================

imputer = SimpleImputer(
    strategy="median"
)

X_train_imputed = imputer.fit_transform(
    X_train
)

X_test_imputed = imputer.transform(
    X_test
)

print(
    "Training matrix:",
    X_train_imputed.shape
)

print(
    "Testing matrix:",
    X_test_imputed.shape
)

Training matrix: (62698, 30)
Testing matrix: (15750, 30)


## 10. Robust Feature Scaling

RobustScaler is fitted only on the training data.

This reduces the influence of extreme sensor values while preventing
information from the test period from entering preprocessing.

In [64]:
# ============================================================
# 10. SCALING
# ============================================================

scaler = RobustScaler()

X_train_scaled = scaler.fit_transform(
    X_train_imputed
)

X_test_scaled = scaler.transform(
    X_test_imputed
)

print(
    "Scaling completed."
)

Scaling completed.


## 11. Baseline — Training Median

A training-median baseline is established before machine learning models
are evaluated.

This provides a simple reference against which the regression models can
be compared.

In [65]:
# ============================================================
# 11. BASELINE
# ============================================================

baseline_prediction = np.full(
    len(y_test),
    y_train.median()
)

baseline_mae = mean_absolute_error(
    y_test,
    baseline_prediction
)

baseline_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        baseline_prediction
    )
)

baseline_r2 = r2_score(
    y_test,
    baseline_prediction
)

print("=" * 70)
print("BASELINE: TRAINING MEDIAN")
print("=" * 70)

print(
    f"MAE  : {baseline_mae:.4f}"
)

print(
    f"RMSE : {baseline_rmse:.4f}"
)

print(
    f"R²   : {baseline_r2:.4f}"
)

BASELINE: TRAINING MEDIAN
MAE  : 1.3503
RMSE : 1.6434
R²   : -2.0786


## 12. Initial Regression Model Comparison

Four tree-based regression algorithms are evaluated using the same
chronological test period:

- Random Forest
- XGBoost
- LightGBM
- CatBoost

At this stage, standard lightweight configurations are used.

The purpose of this stage is to identify the strongest algorithm before
performing targeted hyperparameter tuning.

In [66]:
# ============================================================
# 12. INITIAL MODEL DEFINITIONS
# ============================================================

models = {

    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        max_depth=15,
        random_state=42,
        n_jobs=-1
    ),

    "XGBoost": XGBRegressor(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    ),

    "LightGBM": LGBMRegressor(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        max_depth=-1,
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    ),

    "CatBoost": CatBoostRegressor(
        iterations=300,
        depth=6,
        learning_rate=0.05,
        loss_function="RMSE",
        random_seed=42,
        verbose=False
    )
}

print(
    "Candidate models:",
    list(models.keys())
)

Candidate models: ['Random Forest', 'XGBoost', 'LightGBM', 'CatBoost']


## 13. Initial Model Training and Evaluation

Each candidate model is trained on the training period and evaluated on the
untouched chronological test period.

The metrics used are:

- MAE — lower is better
- RMSE — lower is better
- R² — higher is better

The results will be used to automatically select the best algorithm for
hyperparameter tuning.

In [67]:
# ============================================================
# 13. INITIAL MODEL TRAINING
# ============================================================

initial_results = []

initial_models = {}

for model_name, model in models.items():

    print(
        f"\nTraining {model_name}..."
    )

    start_time = time.time()

    model.fit(
        X_train_scaled,
        y_train
    )

    training_time = (
        time.time()
        -
        start_time
    )

    predictions = model.predict(
        X_test_scaled
    )

    mae = mean_absolute_error(
        y_test,
        predictions
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test,
            predictions
        )
    )

    r2 = r2_score(
        y_test,
        predictions
    )

    initial_models[
        model_name
    ] = model

    initial_results.append({

        "Model":
            model_name,

        "MAE":
            mae,

        "RMSE":
            rmse,

        "R2":
            r2,

        "Training_Time_Seconds":
            training_time
    })

    print(
        f"MAE  : {mae:.4f}"
    )

    print(
        f"RMSE : {rmse:.4f}"
    )

    print(
        f"R²   : {r2:.4f}"
    )

    print(
        f"Time : {training_time:.2f} sec"
    )

initial_results_df = pd.DataFrame(
    initial_results
)

initial_results_df = (
    initial_results_df
    .sort_values(
        by=["R2", "RMSE"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

display(
    initial_results_df
)


Training Random Forest...
MAE  : 0.6601
RMSE : 1.0762
R²   : -0.3203
Time : 1.79 sec

Training XGBoost...
MAE  : 0.6329
RMSE : 0.9940
R²   : -0.1262
Time : 0.54 sec

Training LightGBM...
MAE  : 0.4454
RMSE : 0.8369
R²   : 0.2015
Time : 1.35 sec

Training CatBoost...
MAE  : 0.5935
RMSE : 0.8307
R²   : 0.2134
Time : 0.71 sec


,Model,MAE,RMSE,R2,Training_Time_Seconds
0,CatBoost,0.593516,0.830693,0.213398,0.713913
1,LightGBM,0.445438,0.836930,0.201541,1.348192
2,XGBoost,0.632939,0.993985,-0.126246,0.540291
3,Random Forest,0.660120,1.076199,-0.320259,1.794730


## 14. Automatic Best Model Selection

The best initial model is selected automatically.

The primary selection criterion is the highest R² score.

When two models have similar R² values, lower RMSE is preferred.

The selected algorithm will be the only model subjected to the more
expensive hyperparameter tuning stage.

In [68]:
# ============================================================
# 14. AUTOMATIC BEST MODEL SELECTION
# ============================================================

best_initial_model_name = (
    initial_results_df.iloc[0]["Model"]
)

best_initial_model = initial_models[
    best_initial_model_name
]

best_initial_mae = (
    initial_results_df.iloc[0]["MAE"]
)

best_initial_rmse = (
    initial_results_df.iloc[0]["RMSE"]
)

best_initial_r2 = (
    initial_results_df.iloc[0]["R2"]
)

print("=" * 70)
print("AUTOMATIC BEST MODEL SELECTION")
print("=" * 70)

print(
    "Best initial model:",
    best_initial_model_name
)

print(
    f"MAE  : {best_initial_mae:.4f}"
)

print(
    f"RMSE : {best_initial_rmse:.4f}"
)

print(
    f"R²   : {best_initial_r2:.4f}"
)

AUTOMATIC BEST MODEL SELECTION
Best initial model: CatBoost
MAE  : 0.5935
RMSE : 0.8307
R²   : 0.2134


In [69]:
# ============================================================
# 14A. PERFECT-SCORE LEAKAGE DIAGNOSTIC
# ============================================================

print("=" * 70)
print("PERFECT-SCORE LEAKAGE DIAGNOSTIC")
print("=" * 70)

# Check whether predictions exactly match the target
initial_rf_predictions = initial_models[
    "Random Forest"
].predict(X_test_scaled)

comparison = pd.DataFrame({
    "actual": y_test.values,
    "predicted": initial_rf_predictions
})

comparison["error"] = (
    comparison["actual"]
    -
    comparison["predicted"]
)

print(
    "\nMaximum absolute error:",
    comparison["error"].abs().max()
)

print(
    "Exact prediction matches:",
    (
        comparison["error"].abs() < 1e-12
    ).sum(),
    "/",
    len(comparison)
)

print(
    "\nUnique actual values:",
    sorted(y_test.unique())
)

print(
    "\nUnique predicted values:",
    np.unique(
        np.round(
            initial_rf_predictions,
            6
        )
    )[:20]
)

print(
    "\nTarget distribution in test:"
)

print(
    y_test.value_counts(
        normalize=True
    ).sort_index()
)

PERFECT-SCORE LEAKAGE DIAGNOSTIC

Maximum absolute error: 2.0
Exact prediction matches: 6280 / 15750

Unique actual values: [np.float64(0.0), np.float64(2.0)]

Unique predicted values: [0.   0.01 0.02 0.03 0.04 0.05 0.06 0.07 0.08 0.09 0.1  0.11 0.12 0.14
 0.18 0.19 0.2  0.24 0.25 0.26]

Target distribution in test:
drill_mode
0.0    0.324825
2.0    0.675175
Name: proportion, dtype: float64


## 15. Targeted Hyperparameter Tuning

Only the best-performing initial algorithm is tuned.

This avoids spending unnecessary computation on models that already
underperform the leading candidate.

The tuning process uses a chronological validation split inside the original
training period.

The final chronological test set remains completely untouched and is used
only for final evaluation.

The tuning search is intentionally lightweight to keep training time
reasonable.

In [71]:
# ============================================================
# 15. INTERNAL CHRONOLOGICAL TUNING SPLIT
# ============================================================

tuning_split_index = int(
    len(X_train_scaled) * 0.80
)

X_tune_train = X_train_scaled[
    :tuning_split_index
]

X_tune_valid = X_train_scaled[
    tuning_split_index:
]

y_tune_train = y_train.iloc[
    :tuning_split_index
]

y_tune_valid = y_train.iloc[
    tuning_split_index:
]

print("=" * 70)
print("INTERNAL TUNING SPLIT")
print("=" * 70)

print(
    "Tuning training rows:",
    len(X_tune_train)
)

print(
    "Tuning validation rows:",
    len(X_tune_valid)
)

print(
    "\nTuning validation period:",
    time_train.iloc[
        tuning_split_index
    ],
    "→",
    time_train.iloc[-1]
)

INTERNAL TUNING SPLIT
Tuning training rows: 50158
Tuning validation rows: 12540

Tuning validation period: 2020-11-01 12:00:00 → 2020-11-02 23:00:00


## 16. Automatic Hyperparameter Tuning

The selected model is tuned using a lightweight chronological validation
search.

The tuning configuration is selected automatically based on validation R²,
with RMSE used as a secondary criterion.

Separate parameter spaces are provided for each supported algorithm.

Only the model selected in the previous stage is tuned.

In [72]:
# ============================================================
# 16. AUTOMATIC HYPERPARAMETER TUNING
# ============================================================

tuning_results = []

best_tuned_model = None
best_tuned_params = None

tuning_start_total = time.time()


# ------------------------------------------------------------
# RANDOM FOREST
# ------------------------------------------------------------

if best_initial_model_name == "Random Forest":

    tuning_configs = [

        {
            "n_estimators": 200,
            "max_depth": 10,
            "min_samples_leaf": 1,
            "max_features": 1.0
        },

        {
            "n_estimators": 300,
            "max_depth": 15,
            "min_samples_leaf": 1,
            "max_features": 1.0
        },

        {
            "n_estimators": 400,
            "max_depth": 20,
            "min_samples_leaf": 1,
            "max_features": "sqrt"
        },

        {
            "n_estimators": 300,
            "max_depth": None,
            "min_samples_leaf": 2,
            "max_features": 1.0
        }
    ]


# ------------------------------------------------------------
# XGBOOST
# ------------------------------------------------------------

elif best_initial_model_name == "XGBoost":

    tuning_configs = [

        {
            "n_estimators": 300,
            "max_depth": 4,
            "learning_rate": 0.05,
            "subsample": 0.8,
            "colsample_bytree": 0.8
        },

        {
            "n_estimators": 500,
            "max_depth": 5,
            "learning_rate": 0.05,
            "subsample": 0.8,
            "colsample_bytree": 0.8
        },

        {
            "n_estimators": 800,
            "max_depth": 6,
            "learning_rate": 0.03,
            "subsample": 0.8,
            "colsample_bytree": 0.8
        },

        {
            "n_estimators": 500,
            "max_depth": 7,
            "learning_rate": 0.03,
            "subsample": 0.9,
            "colsample_bytree": 0.9
        }
    ]


# ------------------------------------------------------------
# LIGHTGBM
# ------------------------------------------------------------

elif best_initial_model_name == "LightGBM":

    tuning_configs = [

        {
            "n_estimators": 300,
            "learning_rate": 0.05,
            "num_leaves": 15,
            "max_depth": -1
        },

        {
            "n_estimators": 500,
            "learning_rate": 0.05,
            "num_leaves": 31,
            "max_depth": -1
        },

        {
            "n_estimators": 800,
            "learning_rate": 0.03,
            "num_leaves": 31,
            "max_depth": -1
        },

        {
            "n_estimators": 500,
            "learning_rate": 0.03,
            "num_leaves": 63,
            "max_depth": -1
        }
    ]


# ------------------------------------------------------------
# CATBOOST
# ------------------------------------------------------------

elif best_initial_model_name == "CatBoost":

    tuning_configs = [

        {
            "iterations": 300,
            "depth": 5,
            "learning_rate": 0.05,
            "l2_leaf_reg": 3
        },

        {
            "iterations": 500,
            "depth": 6,
            "learning_rate": 0.05,
            "l2_leaf_reg": 3
        },

        {
            "iterations": 800,
            "depth": 6,
            "learning_rate": 0.03,
            "l2_leaf_reg": 5
        },

        {
            "iterations": 1000,
            "depth": 7,
            "learning_rate": 0.03,
            "l2_leaf_reg": 5
        }
    ]


else:

    raise ValueError(
        f"Unsupported model: "
        f"{best_initial_model_name}"
    )


print(
    f"Tuning {best_initial_model_name}"
)

print(
    f"Configurations to evaluate: "
    f"{len(tuning_configs)}"
)


for config_index, params in enumerate(
    tuning_configs,
    start=1
):

    print(
        f"\nConfiguration "
        f"{config_index}/{len(tuning_configs)}"
    )

    print(params)

    start_time = time.time()


    # --------------------------------------------------------
    # BUILD MODEL
    # --------------------------------------------------------

    if best_initial_model_name == "Random Forest":

        candidate_model = RandomForestRegressor(
            **params,
            random_state=42,
            n_jobs=-1
        )


    elif best_initial_model_name == "XGBoost":

        candidate_model = XGBRegressor(
            **params,
            objective="reg:squarederror",
            random_state=42,
            n_jobs=-1
        )


    elif best_initial_model_name == "LightGBM":

        candidate_model = LGBMRegressor(
            **params,
            random_state=42,
            n_jobs=-1,
            verbosity=-1
        )


    elif best_initial_model_name == "CatBoost":

        candidate_model = CatBoostRegressor(
            **params,
            loss_function="RMSE",
            random_seed=42,
            verbose=False
        )


    # --------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------

    candidate_model.fit(
        X_tune_train,
        y_tune_train
    )


    # --------------------------------------------------------
    # VALIDATION
    # --------------------------------------------------------

    tune_predictions = candidate_model.predict(
        X_tune_valid
    )

    tune_mae = mean_absolute_error(
        y_tune_valid,
        tune_predictions
    )

    tune_rmse = np.sqrt(
        mean_squared_error(
            y_tune_valid,
            tune_predictions
        )
    )

    tune_r2 = r2_score(
        y_tune_valid,
        tune_predictions
    )

    elapsed = (
        time.time()
        -
        start_time
    )

    tuning_results.append({

        "Configuration":
            config_index,

        "MAE":
            tune_mae,

        "RMSE":
            tune_rmse,

        "R2":
            tune_r2,

        "Training_Time_Seconds":
            elapsed,

        **params
    })

    print(
        f"MAE  : {tune_mae:.4f}"
    )

    print(
        f"RMSE : {tune_rmse:.4f}"
    )

    print(
        f"R²   : {tune_r2:.4f}"
    )


# ------------------------------------------------------------
# SELECT BEST CONFIGURATION
# ------------------------------------------------------------

tuning_results_df = pd.DataFrame(
    tuning_results
)

tuning_results_df = (
    tuning_results_df
    .sort_values(
        by=["R2", "RMSE"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

best_tuned_params = {}

for key in tuning_configs[0].keys():

    if key in tuning_results_df.columns:

        best_tuned_params[key] = (
            tuning_results_df.iloc[0][key]
        )

print("\n" + "=" * 70)
print("BEST TUNED CONFIGURATION")
print("=" * 70)

print(
    best_tuned_params
)

display(
    tuning_results_df
)

print(
    "\nTotal tuning time:",
    f"{time.time() - tuning_start_total:.2f}",
    "seconds"
)

Tuning CatBoost
Configurations to evaluate: 4

Configuration 1/4
{'iterations': 300, 'depth': 5, 'learning_rate': 0.05, 'l2_leaf_reg': 3}
MAE  : 0.0535
RMSE : 0.0891
R²   : 0.7542

Configuration 2/4
{'iterations': 500, 'depth': 6, 'learning_rate': 0.05, 'l2_leaf_reg': 3}
MAE  : 0.0346
RMSE : 0.0642
R²   : 0.8724

Configuration 3/4
{'iterations': 800, 'depth': 6, 'learning_rate': 0.03, 'l2_leaf_reg': 5}
MAE  : 0.0483
RMSE : 0.0869
R²   : 0.7662

Configuration 4/4
{'iterations': 1000, 'depth': 7, 'learning_rate': 0.03, 'l2_leaf_reg': 5}
MAE  : 0.0395
RMSE : 0.0672
R²   : 0.8602

BEST TUNED CONFIGURATION
{'iterations': np.float64(500.0), 'depth': np.float64(6.0), 'learning_rate': np.float64(0.05), 'l2_leaf_reg': np.float64(3.0)}


,Configuration,MAE,RMSE,R2,Training_Time_Seconds,iterations,depth,learning_rate,l2_leaf_reg
0,2,0.034633,0.064162,0.872431,0.903317,500,6,0.05,3
1,4,0.039481,0.067163,0.860221,2.169042,1000,7,0.03,5
2,3,0.048287,0.086864,0.766188,1.399815,800,6,0.03,5
3,1,0.053526,0.089066,0.754186,0.468897,300,5,0.05,3



Total tuning time: 4.95 seconds


## 17. Train Final Tuned Model

The best hyperparameter configuration identified using the internal
chronological validation period is now retrained using the complete
training period.

The chronological test period remains untouched until the final evaluation.

In [73]:
# ============================================================
# 17. FINAL TUNED MODEL
# ============================================================

if best_initial_model_name == "Random Forest":

    final_model = RandomForestRegressor(
        **best_tuned_params,
        random_state=42,
        n_jobs=-1
    )


elif best_initial_model_name == "XGBoost":

    final_model = XGBRegressor(
        **best_tuned_params,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )


elif best_initial_model_name == "LightGBM":

    final_model = LGBMRegressor(
        **best_tuned_params,
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    )


elif best_initial_model_name == "CatBoost":

    final_model = CatBoostRegressor(
        **best_tuned_params,
        loss_function="RMSE",
        random_seed=42,
        verbose=False
    )


final_training_start = time.time()

final_model.fit(
    X_train_scaled,
    y_train
)

final_training_time = (
    time.time()
    -
    final_training_start
)

print(
    "Final tuned model trained."
)

print(
    "Training time:",
    f"{final_training_time:.2f} seconds"
)

Final tuned model trained.
Training time: 1.17 seconds


## 18. Final Out-of-Sample Evaluation

The final tuned model is evaluated once on the untouched chronological
test period.

These are the final performance metrics reported for the notebook.

MAE and RMSE should be minimized, while R² should be maximized.

In [74]:
# ============================================================
# 18. FINAL OUT-OF-SAMPLE EVALUATION
# ============================================================

final_predictions = final_model.predict(
    X_test_scaled
)

final_mae = mean_absolute_error(
    y_test,
    final_predictions
)

final_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        final_predictions
    )
)

final_r2 = r2_score(
    y_test,
    final_predictions
)

print("=" * 70)
print("FINAL TUNED MODEL — OUT-OF-SAMPLE RESULTS")
print("=" * 70)

print(
    "Model:",
    best_initial_model_name
)

print(
    "\nBest parameters:"
)

for key, value in best_tuned_params.items():

    print(
        f"  {key:<20}: {value}"
    )

print(
    f"\nMAE  : {final_mae:.4f}"
)

print(
    f"RMSE : {final_rmse:.4f}"
)

print(
    f"R²   : {final_r2:.4f}"
)

print(
    f"\nTraining time: "
    f"{final_training_time:.2f} seconds"
)

FINAL TUNED MODEL — OUT-OF-SAMPLE RESULTS
Model: CatBoost

Best parameters:
  iterations          : 500.0
  depth               : 6.0
  learning_rate       : 0.05
  l2_leaf_reg         : 3.0

MAE  : 0.5881
RMSE : 0.8239
R²   : 0.2262

Training time: 1.17 seconds
